In [ ]:
import pandas as pd
import re
from pathlib import Path

#load data
DATA_DIR = Path.home() 

flights = pd.read_csv(DATA_DIR / "2023_hourly_airport_demand_2023_CLEAN.csv")
weather = pd.read_csv(DATA_DIR / "2023_hourly_weather_data_CLEAN.csv")
load = pd.read_csv(DATA_DIR / "2023_combined_egse_load_data.csv")

# flights = pd.read_csv(DATA_DIR / "2025_hourly_total_flights_data_CLEAN.csv")
# weather = pd.read_csv(DATA_DIR / "2025_hourly_weather_data_correct_padded.csv")
# load = pd.read_csv(DATA_DIR / "2025_combined_egse_load_data.csv")


#standardize necessary column formats
def format_hour(value):
    if pd.isna(value):
        return None

    if isinstance(value, str):
        parts = value.split(":")
        hour = int(parts[0])
    else:
        hour = int(value)

    return f"{hour:02d}:00"
flights["hour"] = flights["Hour"].apply(format_hour)
flights = flights.drop(columns=["Hour"])

load["hour"] = load["Hour"].apply(format_hour)
load = load.drop(columns=["Hour"])

weather = weather.drop(columns=["weather_condition_code"])
weather["hour"] = weather["time"].apply(format_hour)
weather = weather.drop(columns=["time"])


#change date column name
flights = flights.rename(columns={"FlightDate": "date"})
load = load.rename(columns={"FlightDate": "date"})
#weather already uses "date"


#change airport column name 
flights = flights.rename(columns={"Airport": "airport"})
load = load.rename(columns={"Origin": "airport"})
#weather already named correctly

#ensure same type 
flights["airport"] = flights["airport"].astype(str)
load["airport"] = load["airport"].astype(str)
weather["airport"] = weather["airport"].astype(str)
flights["date"] = pd.to_datetime(flights["date"])
load["date"] = pd.to_datetime(load["date"])
weather["date"] = pd.to_datetime(weather["date"])

#load had 5 rows for one time/date/airport section. To merge with 
#other data, we need to make it 1 row so instead of having a column
#listing egse percent with each load(kW) value, I've rearranged it to 
#be a column for each egse percent with the value being the load
#demanded inside. This way, there's only one row for the hour of
#the day per airport per date. 
load_wide = load.pivot_table(
    index=["date", "hour", "airport"],
    columns="eGSE_percent",
    values="Demand_kW",
    aggfunc="mean"
).reset_index()

load_wide = load_wide.rename(columns={
    0.1: "egse_percent_demand_0_1",
    0.25: "egse_percent_demand_0_25",
    0.5: "egse_percent_demand_0_5",
    0.75: "egse_percent_demand_0_75",
    1.0: "egse_percent_demand_1"
})


#clean before merge
flights["hour"] = flights["hour"].astype(str).str.strip()
load_wide["hour"] = load_wide["hour"].astype(str).str.strip()
weather["hour"] = weather["hour"].astype(str).str.strip()

#debugging check duplicates
def check_duplicates(df, name):
    dupes = df.duplicated(subset=["date", "hour", "airport"]).sum()
    print(f"{name} duplicates:", dupes)

check_duplicates(flights, "flights")
check_duplicates(load_wide, "load_wide")
check_duplicates(weather, "weather")

dupes = flights[flights.duplicated(subset=["date", "hour", "airport"], keep=False)]

print(dupes.sort_values(["airport", "date", "hour"]).head(20))


def check_uniqueness(df, name):
    print(f"\n{name}")
    print("rows:", len(df))
    print("unique keys:", df.groupby(["date", "hour", "airport"]).ngroups)
    print("duplicates:", df.duplicated(["date", "hour", "airport"]).sum())

check_uniqueness(flights, "flights")
check_uniqueness(weather, "weather (after hour conversion)")
check_uniqueness(load_wide, "load_wide")

#debugging find what's missing in weather(for 2025 dataset)
keys = ["date", "hour", "airport"]

missing_keys = flights.merge(
    weather[keys].drop_duplicates(),
    on=keys,
    how="left",
    indicator=True
)

print(missing_keys["_merge"].value_counts())

weather_missing = missing_keys[missing_keys["_merge"] == "left_only"]
print(weather_missing.head(20))

#merge
final = flights.merge(load_wide, on=["date", "hour", "airport"])
final = final.merge(weather, on=["date", "hour", "airport"])

print(final.columns)
print(final["hour"].head())

#write to merged csv
#final.to_csv(DATA_DIR / "2025_merged_dataset2.csv", index=False)
final.to_csv(DATA_DIR / "2023_merged_dataset12312.csv", index=False)

C:\Users\alecg\AppData\Local\Temp\ipykernel_15424\1652798735.py:62: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  load["date"] = pd.to_datetime(load["date"])


flights duplicates: 0
load_wide duplicates: 0
weather duplicates: 0
Empty DataFrame
Columns: [date, airport, Departures, Arrivals, TotalFlights, hour]
Index: []

flights
rows: 438000
unique keys: 438000
duplicates: 0

weather (after hour conversion)
rows: 438000
unique keys: 438000
duplicates: 0

load_wide
rows: 438000
unique keys: 438000
duplicates: 0
_merge
both          438000
left_only          0
right_only         0
Name: count, dtype: int64
Empty DataFrame
Columns: [date, airport, Departures, Arrivals, TotalFlights, hour, _merge]
Index: []
Index(['date', 'airport', 'Departures', 'Arrivals', 'TotalFlights', 'hour',
       'egse_percent_demand_0_1', 'egse_percent_demand_0_25',
       'egse_percent_demand_0_5', 'egse_percent_demand_0_75',
       'egse_percent_demand_1', 'temperature', 'humidity', 'precipitation',
       'wind_direction', 'wind_speed', 'pressure', 'cloud_cover'],
      dtype='str')
0    00:00
1    01:00
2    02:00
3    03:00
4    04:00
Name: hour, dtype: str
